# 03 · Modelagem (esquema estrela), Catálogo de Dados e carga → camada gold

**MVP — Onde subsidiar o frete no e-commerce brasileiro: pipeline de dados em nuvem (Databricks)**

Universidade de Brasília (UnB) · Departamento de Engenharia de Produção · Sistemas de Suporte à Decisão
Prof. André Luiz Marques Serrano · **Aluna: Brunna Gavião Moraes — 231025341**


1. Constrói o **esquema estrela**: a fato `fato_item_frete` e as dimensões `dim_tempo`, `dim_produto`, `dim_vendedor` e
   `dim_uf`, esta última com dois papéis, origem e destino.
2. Gera o **Catálogo de Dados** a partir de uma definição única. Dela saem a documentação, o esquema, os comentários e
   as restrições das tabelas.
3. **Valida** os dados contra o catálogo antes da carga.
4. **Carrega** as tabelas na gold com carga idempotente (DROP + CREATE) e aplica as restrições `NOT NULL`, `CHECK`,
   `PRIMARY KEY` e `FOREIGN KEY`.
5. **Confere na nuvem:** contagens, unicidade das chaves, ausência de órfãos e reconciliação silver × gold.

In [0]:
%pip install -q kagglehub statsmodels

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Configuração comum: localiza o repositório, importa o módulo de apoio e detecta a plataforma.
# No Databricks a variável `spark` já existe; fora dele o mesmo código roda localmente (Parquet + DuckDB).
import os, sys, importlib
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
import mvp_frete as mvp
importlib.reload(mvp)

import pandas as pd
pd.set_option("display.max_columns", 50, "display.width", 200)

plat = mvp.Plataforma(spark if "spark" in globals() else None)
print("Ambiente :", plat.preparar())
print("Bronze   :", plat.dir_bronze)
print("Evidências:", plat.dir_evidencias)

Ambiente : Databricks — catálogo workspace, schemas frete_bronze, frete_silver, frete_gold
Bronze   : /Volumes/workspace/frete_bronze/arquivos
Evidências: /Workspace/Users/brunnagmoraes17@gmail.com/MVP3-SSD-Frete-Olist/evidencias


## 1. Leitura da silver e construção do modelo estrela

In [0]:
nomes_silver = ["pedidos", "itens", "clientes", "vendedores", "produtos", "avaliacoes", "geo_cep", "ibge_uf"]
S = {t: plat.ler("silver", t) for t in nomes_silver}
log = mvp.LogLinhagem()
log.linhas = plat.ler("silver", "meta_log_transformacoes").to_dict("records")

G = mvp.construir_estrela(S, log)
display(pd.DataFrame({t: [len(df), df.shape[1]] for t, df in G.items()}, index=["linhas", "colunas"]).T)

linhas,colunas
27,7
608,9
32578,12
3029,7
111752,19


## 2. Modelo e grão

```mermaid
erDiagram
    dim_tempo ||--o{ fato_item_frete : "data da compra"
    dim_produto ||--o{ fato_item_frete : "produto"
    dim_vendedor ||--o{ fato_item_frete : "vendedor"
    dim_uf ||--o{ fato_item_frete : "UF de origem"
    dim_uf ||--o{ fato_item_frete : "UF de destino"
```

* **O grão é o item vendido** (pedido × sequência do item). A Olist registra preço e frete por item; agregar por pedido
  misturaria produtos de categorias e pesos diferentes, e a P3 depende da categoria.
* **`dim_uf` é uma dimensão com papéis:** a mesma tabela responde como origem (vendedor) e destino (cliente).
* **A conciliação com o IBGE está em `dim_uf`:** a população estimada de 2018 entra na dimensão, e assim a P4
  (pedidos por habitante) sai de uma consulta simples.
* **A distância é uma medida da fato,** não uma dimensão, porque depende do par de CEPs do item.

## 3. Catálogo de Dados (documentação gerada a partir da definição única)

In [0]:
cat = mvp.catalogo_df()
(RAIZ / "catalogo").mkdir(exist_ok=True)
try:
    cat.to_csv(RAIZ / "catalogo" / "catalogo_dados.csv", index=False, encoding="utf-8")
    (RAIZ / "catalogo" / "catalogo_dados.md").write_text(mvp.catalogo_markdown(), encoding="utf-8")
    (RAIZ / "scripts" / "ddl_modelo_estrela.sql").write_text(mvp.ddl_script_completo(), encoding="utf-8")
except OSError as e:
    print("Pasta do repositório somente leitura; documentação não regravada:", e)
print(len(cat), "atributos catalogados em", cat.tabela.nunique(), "tabelas")
display(cat)

54 atributos catalogados em 5 tabelas


tabela,atributo,tipo,descricao,dominio,obrigatorio,significado_do_nulo,linhagem
dim_uf,uf,string,Sigla da UF,"AC, AL, AM, AP, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RR, RS, SC, SE, SP, TO",sim,—,"IBGE (código → sigla, R12)"
dim_uf,uf_nome,string,Nome da UF,—,sim,—,"IBGE SIDRA 6579, campo D1N"
dim_uf,codigo_ibge,int,Código IBGE da UF,11 a 53,sim,—,"IBGE SIDRA 6579, D1C"
dim_uf,regiao,string,Grande região,"Norte, Nordeste, Centro-Oeste, Sudeste, Sul",sim,—,Atribuída pela sigla (R12)
dim_uf,populacao_2018,bigint,População residente estimada em 2018 (pessoas),400000 a 50000000,sim,—,"IBGE SIDRA 6579, variável 9324, ano 2018"
dim_uf,lat_centroide,double,Latitude mediana dos CEPs da UF (graus),-34 a 5.3,não,UF sem CEP geolocalizado,Kaggle olist_geolocation (R04)
dim_uf,lon_centroide,double,Longitude mediana dos CEPs da UF (graus),-74 a -34.7,não,UF sem CEP geolocalizado,Kaggle olist_geolocation (R04)
dim_tempo,data_sk,int,Chave da data no formato AAAAMMDD,20170101 a 20180831,sim,—,Gerada (R14)
dim_tempo,data,date,Data,2017-01-01 a 2018-08-31,sim,—,Gerada
dim_tempo,ano,int,Ano,2017 a 2018,sim,—,Derivada de data


## 4. Validação contra o catálogo (antes da carga)

In [0]:
val = mvp.validar_catalogo(G)
mvp.salvar_tabela_evidencia(val, plat, "m1_validacao_catalogo")
display(val)
assert (val.resultado == "OK").all(), "Há atributos fora do domínio do catálogo — carga interrompida."
print("Todos os", len(val), "atributos estão dentro do domínio e da obrigatoriedade declarados.")

tabela,atributo,linhas,nulos,obrigatorio,fora_do_dominio,minimo,maximo,resultado
dim_uf,uf,27,0,true,0,AC,TO,OK
dim_uf,uf_nome,27,0,true,0,Acre,Tocantins,OK
dim_uf,codigo_ibge,27,0,true,0,11,53,OK
dim_uf,regiao,27,0,true,0,Centro-Oeste,Sul,OK
dim_uf,populacao_2018,27,0,true,0,576568,45538936,OK
dim_uf,lat_centroide,27,0,false,0,-29.67298203241743,2.8190192393274165,OK
dim_uf,lon_centroide,27,0,false,0,-67.84012717003617,-35.04121641104507,OK
dim_tempo,data_sk,608,0,true,0,20170101,20180831,OK
dim_tempo,data,608,0,true,0,2017-01-01,2018-08-31,OK
dim_tempo,ano,608,0,true,0,2017,2018,OK


Todos os 54 atributos estão dentro do domínio e da obrigatoriedade declarados.


## 5. Carga na gold (idempotente) e restrições

In [0]:
# DROP + CREATE: reexecutar substitui as tabelas, sem duplicar linhas (fato primeiro, por causa das FKs)
plat.remover_tabelas("gold", list(reversed(mvp.ORDEM_CARGA)))

carga = []
for t in mvp.ORDEM_CARGA:
    n = plat.gravar(G[t], "gold", t, mvp.CATALOGO_GOLD[t]["descricao"], mvp.colunas_catalogo(t))
    carga.append({"tabela": plat.nome_tabela("gold", t), "linhas_validadas": len(G[t]), "linhas_na_nuvem": n,
                  "confere": n == len(G[t])})
carga = pd.DataFrame(carga)
mvp.salvar_tabela_evidencia(carga, plat, "c1_log_carga")
display(carga)
assert carga.confere.all()

tabela,linhas_validadas,linhas_na_nuvem,confere
workspace.frete_gold.dim_uf,27,27,true
workspace.frete_gold.dim_tempo,608,608,true
workspace.frete_gold.dim_produto,32578,32578,true
workspace.frete_gold.dim_vendedor,3029,3029,true
workspace.frete_gold.fato_item_frete,111752,111752,true


In [0]:
# Comentários, NOT NULL, CHECK (domínios do catálogo), PK e FK — só existem no Databricks
resultado_ddl = []
comandos = [c for t in mvp.ORDEM_CARGA for c in mvp.ddl_restricoes(plat, t)] + mvp.ddl_chaves_estrangeiras(plat)
for cmd in comandos:
    try:
        ok = plat.ddl(cmd)
        resultado_ddl.append({"comando": cmd, "status": "aplicado" if ok else "não aplicável (execução local)"})
    except Exception as e:
        resultado_ddl.append({"comando": cmd, "status": f"erro: {str(e).splitlines()[0][:150]}"})
resultado_ddl = pd.DataFrame(resultado_ddl)
TIPOS_DDL = [("COMMENT ON TABLE", "comentário de tabela"), ("SET NOT NULL", "NOT NULL"),
             ("DROP CONSTRAINT", "remoção prévia de CHECK"), ("CHECK (", "CHECK (domínio)"),
             ("PRIMARY KEY", "PRIMARY KEY"), ("FOREIGN KEY", "FOREIGN KEY"), (" COMMENT '", "comentário de coluna")]
resultado_ddl["tipo"] = resultado_ddl.comando.map(lambda c: next(r for k, r in TIPOS_DDL if k in c))
mvp.salvar_tabela_evidencia(resultado_ddl, plat, "c2_restricoes_aplicadas")
display(resultado_ddl.groupby(["tipo", "status"]).size().rename("comandos").reset_index())

tipo,status,comandos
CHECK (domínio),aplicado,41
FOREIGN KEY,aplicado,5
NOT NULL,aplicado,41
PRIMARY KEY,aplicado,5
comentário de coluna,aplicado,54
comentário de tabela,aplicado,5
remoção prévia de CHECK,aplicado,41


In [0]:
# Metadados também persistidos na nuvem: linhagem, catálogo e validação
plat.gravar(log.df(), "gold", "meta_log_transformacoes", "Log de linhagem: regras R01–R16 e linhas afetadas")
plat.gravar(cat, "gold", "meta_catalogo", "Catálogo de Dados do modelo estrela")
plat.gravar(val, "gold", "meta_validacao_catalogo", "Validação dos dados contra o catálogo, antes da carga")
mvp.salvar_tabela_evidencia(log.df(), plat, "m2_log_transformacoes")
display(log.df())

regra,camada_destino,tabela,descricao,linhas_afetadas,linhas_resultantes,executado_em
R01,silver,pedidos/itens,"Tipagem: datas como timestamp, preço e frete como decimal (R$), sequência do item como inteiro",212091,212091,2026-11-09T13:31:38
R02,silver,clientes/vendedores,Prefixo de CEP padronizado com 5 dígitos (zeros à esquerda),0,102536,2026-11-09T13:31:38
R03,silver,clientes/vendedores,UF em maiúsculas e validada nas 27 UFs; cidade em minúsculas sem acento,0,102536,2026-11-09T13:31:38
R04,silver,geo_cep,Descarta coordenadas fora do território brasileiro e resume a geolocalização em um centroide (mediana) por prefixo de CEP,981153,19010,2026-11-09T13:31:40
R05,silver,produtos,Categoria ausente recebe 'sem_categoria' (mantém o item na análise),610,32951,2026-11-09T13:31:40
R06,silver,produtos,Converte g→kg; peso cubado = C×A×L (cm³) ÷ 6000; peso taxado = maior entre peso real e cubado; medidas nulas ou zero viram nulo + flag,6,32951,2026-11-09T13:31:40
R07,silver,avaliacoes,Uma avaliação por pedido: mantém a respondida por último; texto livre descartado (ver R13),551,98673,2026-11-09T13:31:41
R08,silver,pedidos,"Escopo: compras de jan/2017 a ago/2018, exceto pedidos cancelados ou indisponíveis (não geraram envio) e pedidos sem itens",1536,97905,2026-11-09T13:31:41
R13,silver,clientes/avaliacoes,"Minimização (LGPD): customer_unique_id, título e texto das avaliações não seguem para silver e gold",198665,198114,2026-11-09T13:31:41
R12,silver,ibge_uf,Conciliação Olist × IBGE: código IBGE da UF convertido em sigla (chave comum às duas bases); região atribuída pela sigla,0,27,2026-11-09T13:31:41


## 6. Conferência na nuvem, por SQL

In [0]:
plat.usar_gold()
q_unicidade = '''
SELECT 'dim_uf' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT uf) AS chaves_distintas FROM dim_uf
UNION ALL SELECT 'dim_tempo', COUNT(*), COUNT(DISTINCT data_sk) FROM dim_tempo
UNION ALL SELECT 'dim_produto', COUNT(*), COUNT(DISTINCT produto_sk) FROM dim_produto
UNION ALL SELECT 'dim_vendedor', COUNT(*), COUNT(DISTINCT vendedor_sk) FROM dim_vendedor
UNION ALL SELECT 'fato_item_frete', COUNT(*), COUNT(DISTINCT item_sk) FROM fato_item_frete
'''
q_orfaos = '''
SELECT 'data_compra_sk -> dim_tempo' AS chave_estrangeira, COUNT(*) AS orfaos
  FROM fato_item_frete f LEFT JOIN dim_tempo t ON f.data_compra_sk = t.data_sk WHERE t.data_sk IS NULL
UNION ALL SELECT 'produto_sk -> dim_produto', COUNT(*)
  FROM fato_item_frete f LEFT JOIN dim_produto p ON f.produto_sk = p.produto_sk WHERE p.produto_sk IS NULL
UNION ALL SELECT 'vendedor_sk -> dim_vendedor', COUNT(*)
  FROM fato_item_frete f LEFT JOIN dim_vendedor v ON f.vendedor_sk = v.vendedor_sk WHERE v.vendedor_sk IS NULL
UNION ALL SELECT 'uf_origem -> dim_uf', COUNT(*)
  FROM fato_item_frete f LEFT JOIN dim_uf u ON f.uf_origem = u.uf WHERE u.uf IS NULL
UNION ALL SELECT 'uf_destino -> dim_uf', COUNT(*)
  FROM fato_item_frete f LEFT JOIN dim_uf u ON f.uf_destino = u.uf WHERE u.uf IS NULL
'''
q_reconc = f'''
SELECT 'silver.itens' AS origem, COUNT(*) AS itens, ROUND(SUM(preco), 2) AS receita, ROUND(SUM(frete), 2) AS frete
  FROM {plat.nome_tabela("silver", "itens")}
UNION ALL
SELECT 'gold.fato_item_frete', COUNT(*), ROUND(SUM(preco), 2), ROUND(SUM(frete), 2) FROM fato_item_frete
'''
c3 = plat.sql(q_unicidade); c3["chave_unica"] = c3.linhas == c3.chaves_distintas
c4 = plat.sql(q_orfaos)
c5 = plat.sql(q_reconc)
for nome, q in [("c3_unicidade_chaves", q_unicidade), ("c4_integridade_referencial", q_orfaos),
                ("c5_reconciliacao_silver_gold", q_reconc)]:
    mvp.salvar_sql(q, nome)
for nome, df in [("c3_unicidade_chaves", c3), ("c4_integridade_referencial", c4), ("c5_reconciliacao_silver_gold", c5)]:
    mvp.salvar_tabela_evidencia(df, plat, nome); display(df)
assert c3.chave_unica.all() and (c4.orfaos == 0).all() and c5.itens.nunique() == 1
print("Gold conferida: chaves únicas, nenhum órfão e silver × gold reconciliados.")

tabela,linhas,chaves_distintas,chave_unica
dim_uf,27,27,true
dim_tempo,608,608,true
dim_produto,32578,32578,true
dim_vendedor,3029,3029,true
fato_item_frete,111752,111752,true


chave_estrangeira,orfaos
data_compra_sk -> dim_tempo,0
produto_sk -> dim_produto,0
vendedor_sk -> dim_vendedor,0
uf_origem -> dim_uf,0
uf_destino -> dim_uf,0


origem,itens,receita,frete
silver.itens,111752,1.344952968E7,2234177.06
gold.fato_item_frete,111752,1.344952968E7,2234177.06


Gold conferida: chaves únicas, nenhum órfão e silver × gold reconciliados.


> **Evidências a capturar:**
> * *Catalog Explorer → workspace → frete_gold* com as 5 tabelas do modelo e as 3 tabelas `meta_*`
>   (`evidencias/03_gold_tabelas.png`);
> * a aba **Columns** da `fato_item_frete`, com os comentários vindos do catálogo (`evidencias/04_gold_colunas_fato.png`);
> * o **diagrama de relacionamentos** (*View relationships* / *Entity relationship diagram*) gerado pelas PK/FK
>   (`evidencias/05_gold_diagrama_er.png`).